# Lab 1.2 &mdash; The Four Building Blocks

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 1 &middot; Module 1 &mdash; Agents vs. Multi-Agent Systems**

### What you'll do
- Build a <code>@tool</code> that knows what it is not allowed to do
- Bound the history with <code>trim_messages</code> &mdash; and meet the counter that breaks here
- Turn a goal into a typed <code>Plan</code> with <code>with_structured_output</code>
- Assemble all four blocks into one agent

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 1 labs work one case: a small tech-support ticket queue.
> What you build in each lab is picked up by the next one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-1-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# A small tech-support ticket queue. Ordinary rules on purpose: the only new thing in these
# five labs is LangChain. Nothing here is real data and nothing leaves this notebook.

TICKETS = {
    "TCK-4001": {"customer": "Priya Nair",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "VPN-513",
                 "text": "Cannot connect since the upgrade. Error VPN-513."},
    "TCK-4002": {"customer": "Rahul Menon",  "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Monthly export finishes but the PDF is blank."},
    "TCK-4003": {"customer": "Anita Sharma", "product": "Reports",    "version": "3.9.1",
                 "severity": "medium", "error_code": None,
                 "text": "It is just slow today. Nothing else to add."},
    "TCK-4004": {"customer": "Vikram Rao",   "product": "VPN Client", "version": "4.2",
                 "severity": "high",   "error_code": "SEC-900",
                 "text": "Got a login alert from a country I have never visited."},
    "TCK-4005": {"customer": "Priya Nair",   "product": "Reports",    "version": "3.9.1",
                 "severity": "low",    "error_code": "APP-002",
                 "text": "Same blank PDF as my colleague reported."},
}

# The runbook: what support is allowed to do about each error code.
RUNBOOK = {
    "VPN-513": "Certificate pinning changed in 4.2. Have the user clear the local trust store "
               "and re-enrol. Five minutes, no data loss. Support may do this without approval.",
    "APP-002": "Known defect in 3.9.1, fixed in 3.9.2. Advise the upgrade. Do not issue a refund "
               "for this and do not raise a new defect -- link the existing one.",
    "SEC-900": "Possible credential compromise. Escalate to the security desk immediately. "
               "Support must not resolve, close or advise the customer directly.",
}

# Which error codes may an agent resolve on its own, and which need a human?
MUST_ESCALATE = {"SEC-900"}

print(f"{len(TICKETS)} tickets, {len(RUNBOOK)} runbook entries loaded")

## Concept

Every agent is the same four parts. Module 1 gives you each as a real LangChain object.

| Block | The idea | The object |
|---|---|---|
| **LLM** | the reasoning | `ChatOpenAI` |
| **Tools** | the actions | `@tool` |
| **Memory** | what survives a turn | `trim_messages`, a checkpointer |
| **Planning** | a goal is not a sequence | `with_structured_output(...)` |

The one worth slowing down on is **Tools**, because a tool is not a function &mdash; it is a
function *plus what the model is told about it*, and both halves are yours to get right.

## Section 1 &mdash; Tools and memory

A tool that changes something needs to know what it must not change. And a history that grows
without bound will eventually push your instructions out of the window.

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages
from langchain_core.messages.utils import count_tokens_approximately

SYSTEM = ("You are a tech support analyst. Answer only from the data the tools give you. "
          "If the runbook says escalate, say so and stop.")


@tool
def resolve_ticket(ref: str, resolved_by: str) -> str:
    """Close a support ticket. Only call this once the runbook says support may act.

    Requires the name of the person taking responsibility for the resolution.
    """
    code = TICKETS.get(ref, {}).get("error_code")
    if code in MUST_ESCALATE:      # the tool refuses; it does not rely on the model behaving
        return f"refused: {code} must go to the security desk, not be resolved here"
    return f"{ref} resolved by {resolved_by}"


def bounded(messages: list, max_tokens: int = 120) -> list:
    """Keep the system message and as many recent turns as fit."""
    # token_counter=<your chat model> is what everyone writes first, and it raises
    # NotImplementedError here: there is no tiktoken encoding for this model.
    with_the_model     = get_llm                   # referenced, not called
    with_a_plain_count = count_tokens_approximately

    return trim_messages(messages, max_tokens=max_tokens,
                         token_counter=with_a_plain_count,
                         strategy="last", include_system=True,
                         start_on="human", allow_partial=False)

In [ ]:
# --- Self-check: Section 1   (a real @tool and a real trim -- neither calls the model)
def long_history():
    msgs = [SystemMessage(SYSTEM), HumanMessage("Look at TCK-4001.")]
    for i in range(12):
        msgs += [AIMessage(f"step {i}: " + "checking the ticket. " * 12), HumanMessage(f"and then? ({i})")]
    return msgs

check("resolve_ticket refuses the security ticket even when told to close it",
      lambda: "refused" in resolve_ticket.invoke({"ref": "TCK-4004", "resolved_by": "Dev"}),
      "the runbook says escalate -- so the TOOL enforces it, rather than hoping the model read it")
check("it still resolves an ordinary ticket",
      lambda: "resolved by" in resolve_ticket.invoke({"ref": "TCK-4001", "resolved_by": "Dev"}))
check("the history is bounded",
      lambda: count_tokens_approximately(bounded(long_history())) <= 130,
      "token_counter=get_llm() raises NotImplementedError here -- there is no tiktoken encoding "
      "for this model, so the counter has to be a plain function over the text")
check("trimming kept the instructions and dropped old turns",
      lambda: (isinstance(bounded(long_history())[0], SystemMessage)
               and len(bounded(long_history())) < len(long_history())))
score()

## Section 2 &mdash; Planning, and all four together

`with_structured_output(Plan)` makes the model return a **`Plan` object**, not prose you then have
to parse.

The `Field(description=...)` lines are the part that matters and the part everyone treats as
documentation. They are not documentation: LangChain sends them to the model **as the schema**.
They are the only instruction it gets about what belongs in each field.

In [ ]:
from typing import List
from pydantic import BaseModel, Field


class Step(BaseModel):
    """One step of a support investigation."""
    name: str = Field(description="A short imperative label, e.g. 'read the ticket'")
    tool: str = Field(description="One of: lookup_ticket, runbook_for, resolve_ticket, none")


class Plan(BaseModel):
    """An ordered plan for handling one support ticket."""
    goal: str = Field(description="The question this plan answers, in one line")
    steps: List[Step] = Field(description="The steps, in the order they should run")

In [ ]:
# --- Self-check: Section 2   (the schema object, before any model sees it)
def described(field):
    """The description the model will be sent. Unfilled blanks are still the literal 'BLANK'."""
    d = Step.model_fields[field].description
    if d.strip() == "BLANK":
        raise NameError(f"Step.{field} description is still BLANK")   # -> [TODO], not [FAIL]
    return d

check("the step name description says what a name should look like",
      lambda: len(described("name")) > 20)
check("the tool description names the allowed values",
      lambda: sum(t in described("tool") for t in ("lookup_ticket", "runbook_for", "none")) >= 2,
      "the model cannot pick from a list it was never shown")
def _rejects():
    """Pydantic must object to a step with no tool."""
    try:
        Plan(goal="g", steps=[{"name": "read"}])
        return False
    except Exception:
        return True

check("Plan validates a well-formed plan and rejects a malformed one",
      lambda: bool(Plan(goal="g", steps=[Step(name="read the ticket", tool="lookup_ticket")]))
              and _rejects())
score()

## Run it for real &mdash; all four blocks, one agent

In [ ]:
if llm_ready():
    plan = guard(lambda: get_llm().with_structured_output(Plan).invoke(
        "Plan how to handle support ticket TCK-4004. Use only the tools named in the schema."))
    if plan:
        print("goal:", plan.goal)
        for s in plan.steps:
            print(f"   {s.name:38} -> {s.tool}")

In [ ]:
def assemble_and_run():
    from langchain.agents import create_agent
    from langgraph.checkpoint.memory import InMemorySaver

    @tool
    def lookup_ticket(ref: str) -> str:
        """Return the support ticket for one reference such as 'TCK-4001'."""
        t = TICKETS.get(ref)
        return json.dumps({"ref": ref, **t}) if t else f"no ticket {ref!r}"

    @tool
    def runbook_for(error_code: str) -> str:
        """Return what support is allowed to do about one error code, e.g. 'VPN-513'."""
        return RUNBOOK.get(error_code, f"no runbook entry for {error_code!r}")

    agent = create_agent(model=get_llm(),                                     # LLM
                         tools=[lookup_ticket, runbook_for, resolve_ticket],  # Tools
                         system_prompt=SYSTEM,                                # Planning, of a sort
                         checkpointer=InMemorySaver())                        # Memory

    for ref in ["TCK-4001", "TCK-4004"]:
        out = agent.invoke({"messages": [("user", f"Handle {ref} end to end.")]},
                           {"configurable": {"thread_id": ref}})
        print(f"{ref}: {out['messages'][-1].content[:200]}\n")

if llm_ready():
    guard(assemble_and_run)      # resolve_ticket carries a blank -- the agent INVOKES it

### Read it

**`TCK-4001` was resolved and `TCK-4004` was not** &mdash; and the difference was not the model
being careful. `resolve_ticket` refuses, in Python, whatever it is asked. That is the whole idea
behind Module 8: a guardrail the model can talk its way past is not a guardrail.

**The token counter is a real trap.** `token_counter=get_llm()` is the obvious thing to write and
it raises `NotImplementedError` here &mdash; there is no tiktoken encoding for this model.
`count_tokens_approximately` is a plain function over the message text and works everywhere.

**The `Field` descriptions are prompt engineering.** You wrote them as documentation and the model
read them as instructions. Lab 1.3 measures exactly how much that is worth.

In [ ]:
score()

## Your turn

1. Set `max_tokens=40` in `bounded` and print what survives. At what point do your instructions
   fall out of the window? That number is when your agent starts ignoring its system prompt.
2. Change `Step.tool`'s description to just `"the tool"` and re-run the plan. Read what the model
   puts there now.